Benchmark questions and ground-truth SQL for the Genie agent over the dbt marts (`mart_account_360`, `mart_account_monthly_health`) and Gold tables. Expected values are not stored: results depend on the data and the run date, so the SQL is the reference.

Last successful run: 10 of 10 on 27th Oct 2026 (the set the instructions were tuned against).

##### 1. How many accounts have at least one escalated case?

In [0]:
%sql
SELECT
  COUNT(*)
FROM
  salesforce_dev.dbt_dev.mart_account_360
WHERE
  NOT is_unknown_account
  AND cases_escalated > 0;

##### 2. How many accounts have no commitment this month?

In [0]:
%sql
SELECT
  COUNT(*)
FROM
  salesforce_dev.dbt_dev.mart_account_monthly_health
WHERE
  NOT is_unknown_account
  AND commit_status = 'no_commitment'
  AND month_start
    = (
      SELECT
        MAX(month_start)
      FROM
        salesforce_dev.dbt_dev.mart_account_monthly_health
    );

##### 3. How many accounts increased usage by more than 50% on the previous month, in the latest month?

In [0]:
%sql
SELECT
  COUNT(*)
FROM
  salesforce_dev.dbt_dev.mart_account_monthly_health
WHERE
  NOT is_unknown_account
  AND usage_mom_change_pct > 50
  AND month_start
    = (
      SELECT
        MAX(month_start)
      FROM
        salesforce_dev.dbt_dev.mart_account_monthly_health
    );

##### 4. How many Platinum-tier accounts are on track this month?

In [0]:
%sql
SELECT
  COUNT(*)
FROM
  salesforce_dev.dbt_dev.mart_account_monthly_health
WHERE
  NOT is_unknown_account
  AND contract_tier = 'Platinum'
  AND commit_status = 'on_track'
  AND month_start
    = (
      SELECT
        MAX(month_start)
      FROM
        salesforce_dev.dbt_dev.mart_account_monthly_health
    );

##### 5. What is the total lifetime usage in AUD, excluding the UNKNOWN account?

In [0]:
%sql
SELECT
  SUM(lifetime_usage_aud)
FROM
  salesforce_dev.dbt_dev.mart_account_360
WHERE
  NOT is_unknown_account;

##### 6. Which 5 accounts have the largest dollar shortfall this month?

In [0]:
%sql
SELECT
  account_name,
  variance_aud
FROM
  salesforce_dev.dbt_dev.mart_account_monthly_health
WHERE
  NOT is_unknown_account
  AND variance_aud is not NULL
  AND month_start
    = (
      SELECT
        MAX(month_start)
      FROM
        salesforce_dev.dbt_dev.mart_account_monthly_health
    )
ORDER BY
  variance_aud ASC
LIMIT 5;

##### 7. What are the top 5 industries by escalation rate?

In [0]:
%sql
SELECT
  industry,
  SUM(cases_escalated) AS total_escalated,
  SUM(cases_opened) AS total_opened,
  SUM(cases_escalated) / SUM(cases_opened) AS escalation_rate
FROM
  salesforce_dev.dbt_dev.mart_account_monthly_health
WHERE
  NOT is_unknown_account
  AND industry IS NOT NULL
GROUP BY
  industry
HAVING
  SUM(cases_opened) > 0
ORDER BY
  escalation_rate DESC
LIMIT 5;

##### 8. How many accounts are in each commit status this month?

In [0]:
%sql
SELECT
  commit_status,
  COUNT(*) AS accounts
FROM
  salesforce_dev.dbt_dev.mart_account_monthly_health
WHERE
  NOT is_unknown_account
  AND month_start
    = (
      SELECT
        MAX(month_start)
      FROM
        salesforce_dev.dbt_dev.mart_account_monthly_health
    )
GROUP BY
  commit_status;

##### 9. How many known accounts are there?

In [0]:
%sql
SELECT
  COUNT(*)
FROM
  salesforce_dev.dbt_dev.mart_account_360
WHERE
  NOT is_unknown_account;

##### 10. How many accounts are under-utilising their commitment this month?

In [0]:
%sql
SELECT
  COUNT(*) AS under_accounts
FROM
  salesforce_dev.dbt_dev.mart_account_monthly_health
WHERE
  commit_status = 'under'
  AND NOT is_unknown_account
  AND month_start
    = (
      SELECT
        MAX(month_start)
      FROM
        salesforce_dev.dbt_dev.mart_account_monthly_health
    );